# Extract source bead frames

Run this **at the source microscope**. Each source acquisition has ~30 frames,
but only the fiducial (bead) frames are needed to measure cross-microscope drift.
This notebook writes a compact per-FOV `.tiff` holding only the **full-resolution
bead frames** (≈2 frames vs ~30), plus a matching compact frame table — so you can
copy just those small files to the NAS and run **Part 2** of
`align_fovs_across_microscopes.ipynb` on the target microscope.

The bead frames are kept at **full resolution** on purpose: the drift step uses
`phase_cross_correlation`, which correlates the image pixels (it needs the whole
frame, not fitted bead positions).

**Downstream wiring** — in `align_fovs` Part 2, point:
- `SOURCE_BEAD_DIR` → this `OUTPUT_DIR` (after moving it to the NAS)
- `SOURCE_BEAD_PATTERN` → `OUTPUT_PATTERN` below (a `.tiff`)
- `SOURCE_FRAME_TABLE` → the compact frame table written here

In [ ]:
import os
import sys
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.alignment import bead_frame_indices, extract_bead_frames
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress_display      import ProgressReporter

# SAMPLE_NAME/IMAGING_DIR resolved from folder structure, NOT SAMPLE_DIR.name --
# this notebook doesn't build any positions_*.txt filename itself, but
# POSITIONS_TAG is derived here anyway for consistency with the rest of the
# pipeline (and in case a future edit adds one).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR  : {SAMPLE_DIR}")
print(f"SAMPLE_NAME : {SAMPLE_NAME}")
print(f"POSITIONS_TAG : {POSITIONS_TAG}")

In [ ]:
# ── Parameters (run at the SOURCE microscope) ───────────────────────
SOURCE_LABEL = "mf4"

# Where the source acquisitions live, and the per-FOV filename pattern
# (Python format with {fov}; WITHOUT the file-type suffix).
SOURCE_DATA_DIR = SAMPLE_DIR / "data" / "cells"                  # <-- edit
SOURCE_PATTERN  = f"hal-{SOURCE_LABEL}-cells_{{fov:03d}}"    # <-- edit
IMAGE_SUFFIX    = ".dax"     # ".dax" / ".zarr" / ".tiff"
FRAME_W = None              # required for .dax (e.g. 2048); None for .zarr/.tiff
FRAME_H = None

# Frame table for the source series (used to locate the bead frames).
SOURCE_FRAME_TABLE = SAMPLE_DIR / "metadata" / "frame_table_405f25-488f2-seq.csv"  # <-- edit
BEAD_COLOR = None           # None = auto-detect the single-z fiducial colour (e.g. 488)

# Output: compact bead files. Point this at the NAS, or write locally and move it.
OUTPUT_DIR     = SAMPLE_DIR / "beads_source"                     # <-- edit (e.g. a NAS path)
OUTPUT_PATTERN = f"beads_{SOURCE_LABEL}_{{fov:03d}}.tiff"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source data dir : {SOURCE_DATA_DIR}")
print(f"Output dir      : {OUTPUT_DIR}")

In [ ]:
# ── Determine the bead frames and write the compact frame table ─────
src_ft   = pd.read_csv(SOURCE_FRAME_TABLE, index_col=0)
bead_idx = bead_frame_indices(src_ft, bead_color=BEAD_COLOR)
print(f"Bead frames in source series: {bead_idx} (colour {src_ft.loc[bead_idx[0], 'color']})")

# Compact frame table (just the bead rows, reindexed 0..k) for align_fovs Part 2,
# so select_bead_frame resolves correctly against the compact .tiff files.
compact_ft      = src_ft.loc[bead_idx].reset_index(drop=True)
compact_ft_path = OUTPUT_DIR / f"frame_table_beads_{SOURCE_LABEL}.csv"
compact_ft.to_csv(compact_ft_path)
print(f"Wrote compact frame table: {compact_ft_path}")

In [ ]:
# ── Extract bead frames for every source FOV (in parallel) ───────────────────
prefix    = SOURCE_PATTERN.split("{")[0]                 # literal text before {fov}
fov_regex = re.compile(re.escape(prefix) + r"(\d+)")
src_files = sorted(SOURCE_DATA_DIR.glob(prefix + "*" + IMAGE_SUFFIX))
print(f"Found {len(src_files)} source file(s) matching '{prefix}*{IMAGE_SUFFIX}'.")

jobs = []   # (src_path, out_path) pairs -- built up front so the pool below only submits real work
for f in src_files:
    m = fov_regex.search(f.name)
    if not m:
        continue
    fov = int(m.group(1))
    jobs.append((f, OUTPUT_DIR / OUTPUT_PATTERN.format(fov=fov)))

n_workers = max(1, (os.cpu_count() or 2) - 2)   # same default convention as ExperimentConfig.resolved_n_workers
tot_in = tot_out = 0
reporter = ProgressReporter(total=len(jobs), label="Extracting bead frames")
with ProcessPoolExecutor(max_workers=n_workers) as pool:
    futures = {
        pool.submit(extract_bead_frames, f, out, bead_idx, frame_width=FRAME_W, frame_height=FRAME_H): (f, out)
        for f, out in jobs
    }
    for future in as_completed(futures):
        f, out = futures[future]
        future.result()   # re-raise here if a worker errored on this FOV
        tot_in  += f.stat().st_size if f.is_file() else 0
        tot_out += out.stat().st_size
        reporter.update()
reporter.done()
n = len(jobs)

print(f"Extracted {n} FOV(s) -> {OUTPUT_DIR}")
if tot_out and tot_in:
    print(f"Total size: {tot_in/1e6:.1f} MB -> {tot_out/1e6:.1f} MB "
          f"({tot_in/tot_out:.1f}x smaller)")
print("\nNext: move OUTPUT_DIR to the NAS, then run align_fovs Part 2 on the target scope\n"
      "with SOURCE_BEAD_DIR=OUTPUT_DIR, SOURCE_BEAD_PATTERN=OUTPUT_PATTERN, "
      "SOURCE_FRAME_TABLE=the compact frame table.")